# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

# preprocesador para árboles: solo imputación por mediana
tree_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), feature_columns)
    ]
)

# preprocesador para modelos sensibles a la escala: imputación + StandardScaler
scaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), feature_columns)
    ]
)


**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

> Porque si el scaler se ajusta antes de la validación cruzada, calcula la media y la desviación con todos los datos, incluidos los que luego se usan para validar en cada fold. Así la validación ya conoce información del conjunto que debería ser nuevo y el resultado sale demasiado optimista, Dentro del pipeline, en cada fold el scaler se ajusta solo con los datos de entrenamiento de ese fold.

## 3. Modelos candidatos

In [5]:
# crea pipelines para DecisionTree, LogisticRegression y KNN
models = {
    "DecisionTree": Pipeline([
        ("prep", clone(tree_preprocessor)),
        ("model", DecisionTreeClassifier(random_state=RANDOM_STATE)),
    ]),
    "LogisticRegression": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN": Pipeline([
        ("prep", clone(scaled_preprocessor)),
        ("model", KNeighborsClassifier()),
    ]),
}


## 4. Comparación inicial con validación cruzada

In [6]:
# usa StratifiedKFold de 5 folds y cross_validate
# Reporta accuracy promedio de train, accuracy promedio de validation y desviación estándar
# Guarda el resumen en initial_comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

rows = []
for name, pipe in models.items():
    scores = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring="accuracy", return_train_score=True
    )
    rows.append({
        "model": name,
        "train_accuracy": scores["train_score"].mean(),
        "val_accuracy": scores["test_score"].mean(),
        "val_std": scores["test_score"].std(),
    })

initial_comparison = (
    pd.DataFrame(rows)
    .sort_values("val_accuracy", ascending=False)
    .reset_index(drop=True)
)
print(initial_comparison.round(4))


                model  train_accuracy  val_accuracy  val_std
0  LogisticRegression          1.0000        0.9791   0.0277
1                 KNN          0.9771        0.9507   0.0353
2        DecisionTree          1.0000        0.9022   0.0586


## 5. Optimización de hiperparámetros

In [7]:
# define espacios pequeños de búsqueda para los tres modelos
param_grids = {
    "DecisionTree": {
        "model__max_depth": [2, 3, 4, 5, None],
        "model__min_samples_leaf": [1, 3, 5],
    },
    "LogisticRegression": {
        "model__C": [0.01, 0.1, 1, 10],
    },
    "KNN": {
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
    },
}

# ejecuta GridSearchCV con scoring="accuracy"
searches = {}
rows = []

for name, pipe in models.items():
    search = GridSearchCV(
        pipe, param_grids[name],
        cv=cv, scoring="accuracy", return_train_score=True
    )
    search.fit(X_train, y_train)
    searches[name] = search

    # métricas de la mejor combinación
    best = search.best_index_
    rows.append({
        "model": name,
        "train_accuracy": search.cv_results_["mean_train_score"][best],
        "val_accuracy": search.best_score_,
        "val_std": search.cv_results_["std_test_score"][best],
        "best_params": search.best_params_,
    })

tuned_comparison = (
    pd.DataFrame(rows)
    .sort_values("val_accuracy", ascending=False)
    .reset_index(drop=True)
)
print(tuned_comparison.round(4))


                model  train_accuracy  val_accuracy  val_std  \
0  LogisticRegression          1.0000        0.9791   0.0277   
1                 KNN          0.9648        0.9579   0.0405   
2        DecisionTree          0.9982        0.9022   0.0586   

                                         best_params  
0                                    {'model__C': 1}  
1  {'model__n_neighbors': 7, 'model__weights': 'u...  
2  {'model__max_depth': 4, 'model__min_samples_le...  


## 6. Selección y persistencia del campeón

In [8]:
# selecciona el mayor best_score_ de CV
champion_name = max(searches, key=lambda name: searches[name].best_score_)
champion = searches[champion_name].best_estimator_

print("Campeón:", champion_name)
print("Parámetros:", searches[champion_name].best_params_)
print("Accuracy CV:", round(searches[champion_name].best_score_, 4))

# guarda champion_model.joblib, training_metadata.json y cv_results.csv
joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "best_params": searches[champion_name].best_params_,
    "cv_accuracy": float(searches[champion_name].best_score_),
    "cv_folds": 5,
    "random_state": RANDOM_STATE,
    "features": feature_columns,
    "target": contract["target"],
}

(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2),
    encoding="utf-8"
)

cv_results = pd.concat([
    initial_comparison.assign(stage="initial"),
    tuned_comparison.assign(stage="tuned"),
])
cv_results.to_csv(REPORTS_DIR / "cv_results.csv", index=False)


Campeón: LogisticRegression
Parámetros: {'model__C': 1}
Accuracy CV: 0.9791


In [9]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

Challenge 2 completado.


**Justificación de la selección:**

> Se eligio LogisticRegression con `C=1`.


- Desempeño: tiene la mayor accuracy en validación cruzada (0.979)

- Variabilidad: es también la más estable entre folds (desviación de 0.028, contra 0.041 de KNN y 0.059 del árbol).

- Sobreajuste: acierta el 100 % en train, pero la diferencia con validación es de unos 2 puntos. El árbol sobreajusta mas (casi 10 puntos), KNN tiene la brecha más pequeña, pero su validación es peor.

- Complejidad: es simple, con un solo hiperparámetro que ajustar, y KNN tiene que guardar todos los datos para predecir.

-*Interpretabilidad: sus coeficientes muestran cómo pesa cada variable en cada clase


## Checklist

- [ x] Todo el procesamiento está dentro de pipelines.
- [ x] Comparé tres familias de modelos.
- [ x] Optimicé hiperparámetros con CV.
- [ x] Elegí el campeón sin usar test.
- [ ]x Guardé el pipeline completo y sus metadatos.